# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`

This notebook provides a step-by-step guide for loading and exploring the FAIR² dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library. All dataset elements, including record sets and fields, are referenced by their Croissant `@id`.

### Dataset Source
The dataset's Croissant schema is available at:
```
https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json
```

In [ ]:
# Install mlcroissant if not already installed
!pip install mlcroissant --quiet

## 1. Data Loading

We first load the dataset metadata and explore the top-level description using `mlcroissant`.


In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the dataset schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json"

# Load the Croissant dataset
dataset = mlc.Dataset(croissant_url)

# Display the dataset metadata summary
meta = dataset.metadata
print(f"{meta.name}\n{'='*len(meta.name)}\n{meta.description}\n")
print(f"Dataset DOI: {getattr(meta, 'identifier', None)}")
print(f"Version: {getattr(meta, 'version', None)}")
print(f"Published: {getattr(meta, 'datePublished', None)}")

## 2. Data Overview

Next, we'll enumerate the available **record sets**, their `@id`s, and associated fields (all by `@id`). This allows us to reference and extract subsets with precise control.

In [ ]:
# List all record sets (tables) in the dataset with their @id
print("Available Record Sets (@id):")
for rs in dataset.record_sets():
    print(f"- {rs['@id']}  (name: {rs.get('name', '')})")

# Let's pick the main (likely only) data record set
record_sets = [rs['@id'] for rs in dataset.record_sets()]
if len(record_sets) == 0:
    raise RuntimeError("No record sets were found in the dataset.")

# For each record set, print out all field @id and field properties
for rs in dataset.record_sets():
    print(f"\nRecord Set: {rs['@id']}")
    print("Fields (@id, name, dataType):")
    for field in rs.get('field', []):
        # field may be a string (just @id) or dict with @id, name, etc.
        if isinstance(field, dict):
            print(f"  - {field.get('@id')} (name: {field.get('name')}, type: {field.get('dataType', '')})")
        else:
            print(f"  - {field}")

## 3. Data Extraction

Let's extract the data for the available record set(s) into Pandas DataFrames. We continue to reference record sets and fields by `@id` only.

In [ ]:
# Extract all data from each record set as a DataFrame
dataframes = {}

for rs_id in record_sets:
    # records is a generator of dicts where keys are field @id
    records = list(dataset.records(record_set=rs_id))
    df = pd.DataFrame(records)
    dataframes[rs_id] = df

# Show the columns (field @id) in the first record set
primary_rs = record_sets[0]
print(f"\nFields (@id) in record set {primary_rs}:")
print(dataframes[primary_rs].columns.tolist())

print("\nPreview of extracted data:")
dataframes[primary_rs].head()

## 4. Exploratory Data Analysis (EDA)

For analysis, let's select a numeric field (for example: patient age, interval years, or similar; you'll need to replace the variable below with the appropriate `@id` from the previous output). We'll filter, normalize, and group records using that field. This entire process always uses the field's `@id`.

In [ ]:
# Choose main dataframe and numeric field
df = dataframes[primary_rs]

# Example: select a numeric field from the dataset (update as needed)
# By printing field @ids in the previous step, we can choose a numeric column: let's try to select by standard names.
possible_numeric_fields = [col for col in df.columns if 'age' in col.lower() or 'interval' in col.lower() or 'year' in col.lower() or df[col].dtype in ['int64', 'float64', 'int32', 'float32']]
if len(possible_numeric_fields) == 0:
    print("No obvious numeric columns found. Please edit 'numeric_field_id' below to match an integer or float field @id from earlier.")
    numeric_field_id = df.columns[0]  # fallback
else:
    numeric_field_id = possible_numeric_fields[0]
print(f"Selected numeric field for EDA: {numeric_field_id}")

# Set a filter threshold (update as needed for your data)
threshold = df[numeric_field_id].quantile(0.5) if pd.api.types.is_numeric_dtype(df[numeric_field_id]) else 0

filtered_df = df[df[numeric_field_id] > threshold]
print(f"\nFiltered records from '{primary_rs}' where '{numeric_field_id}' > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field for filtered data
norm_col = f"{numeric_field_id}_normalized"
filtered_df[norm_col] = (filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()) / filtered_df[numeric_field_id].std()

print(f"\nNormalized '{numeric_field_id}' for filtered records:")
display(filtered_df[[numeric_field_id, norm_col]].head())

# Try grouping by another field, e.g., sex, anatomical_location, etc.
possible_group_fields = [col for col in df.columns if 'sex' in col.lower() or 'site' in col.lower() or 'group' in col.lower() or 'location' in col.lower() or 'type' in col.lower()]
group_field_id = possible_group_fields[0] if possible_group_fields else None
if group_field_id:
    grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index().sort_values(numeric_field_id, ascending=False)
    print(f"\nGrouped mean '{numeric_field_id}' by '{group_field_id}':")
    display(grouped_df.head())
else:
    print("No categorical field found for grouping. Please edit 'group_field_id'.")

## 5. Visualization

Let's visualize the distribution of our selected numeric field (e.g., histogram) and, if grouped, a bar plot of group means.


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the numeric field
plt.figure(figsize=(7, 4))
sns.histplot(df[numeric_field_id].dropna(), bins=15, kde=True, color='skyblue')
plt.title(f"Distribution of {numeric_field_id}")
plt.xlabel(numeric_field_id)
plt.ylabel("Count")
plt.show()

# Barplot of mean numeric field by group (if available)
if group_field_id:
    plt.figure(figsize=(8, 4))
    sns.barplot(data=grouped_df, x=group_field_id, y=numeric_field_id, palette="Set2")
    plt.title(f"Mean {numeric_field_id} by {group_field_id}")
    plt.ylabel(f"Mean {numeric_field_id}")
    plt.xlabel(group_field_id)
    plt.xticks(rotation=30)
    plt.tight_layout()
    plt.show()

## 6. Conclusion

In this notebook, we've demonstrated how to load and explore the FAIR² dataset's record sets and fields—strictly using their Croissant `@id`—with `mlcroissant`.

- **Metadata & Structure:** Loaded rich dataset metadata and observed data structure by record set/field `@id`.
- **Data Extraction:** Parsed the main tabular record set into a DataFrame, referencing all fields by `@id`.
- **EDA:** Applied essential analysis steps: numerical filtering, normalization, and grouping on chosen fields, always by `@id`.
- **Visualization:** Generated distribution and group summary plots for variable understanding.

This notebook can be extended for deeper statistical analysis, modeling, or export of clean subsets. For other record sets, simply repeat the extraction step, always indexing by their `@id`.

---

**References:**
- [FAIR² Croissant Dataset Schema](https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json)
- [`mlcroissant` documentation](https://github.com/mlcommons/croissant)